# Radar line-of-sight velocity maps

Edit the settings below, then run all cells using the `dff` conda kernel. Requires numpy, pandas, matplotlib, cartopy, pyproj, pydarn and pydarnio (install Jupyter/ipykernel to run interactively). Reads local `data/YYYYMMDD/RADAR/*.fitacf[3].bz2`; no DFF or MAP2 file is needed.

Each map contains all observations in a half-open time window `[start, end)`, without temporal averaging. The final window is clipped to the requested end. Times are UT; an end earlier than the start means the following day. Equal start/end times are rejected. Ground scatter (`gflg=1`) uses circles; ionospheric scatter (`gflg=0`) uses dots. Unknown flags are excluded. Positive velocity points **toward** the radar; negative velocity points away. These are measured LOS components, not reconstructed horizontal flow vectors.

Positions use pyDARN's geographic gate centers and standard virtual-height propagation model, with date-specific hardware and each record's range settings. Ground scatter uses the same positioning model: positions are approximate and are not a dedicated ground-scatter range inversion. Zero-speed returns have markers but no visible arrows. Both scatter types share a fixed velocity color scale across all maps.

Maps are arranged in one grid with a shared velocity colorbar and saved as a single PNG. Set `GRID_COLUMNS` to change the layout.

Coordinate selection: `COORDINATES="aacgmv2"` (default) shows AACGM-v2 latitude/longitude with transformed coastlines, `"aacgmv2_mlt"` shows magnetic latitude versus MLT, and `"geographic"` restores the geographic map. The local `sd_carto.py` utility was copied from `meso_tracers/plotting_utils/sd_carto.py`. Each panel uses its window midpoint for all coordinate conversions, including coastlines and radar sites. Geographic arrow bases and endpoints are both converted before projection. Magnetic grid labels are magnetic degrees (or MLT hours). Set the extent for the chosen coordinate system independently. All conversions use `AACGM_HEIGHT_KM`; radar markers are reference positions at that conversion altitude. Saved files are separated by coordinate mode.


In [ ]:
EVENT_DATE = "2011-04-02"
START_TIME = "05:00"
END_TIME = "06:00"
TIME_DELTA = "5min"  # pandas duration string or datetime.timedelta
RADARS = ["cve", "cvw", "bks", "fhw", "fhe"]
SCATTER = "both"  # "ground", "ionospheric", or "both"

DATA_ROOT = "data"
SAVE_MAPS = True
OUTPUT_ROOT = "outputs/los_maps"
MAP_EXTENT = [-130, -45, 30, 85]  # west, east, south, north (geographic degrees)
VELOCITY_LIMIT = 1000  # m/s; colors saturate beyond +/- this value
VECTOR_SECONDS = 120  # display scale: 1000 m/s arrow spans 120 km
COASTLINES = True  # first use may download Cartopy Natural Earth coastlines

GRID_COLUMNS = 3  # number of maps per row

COORDINATES = "aacgmv2"  # "aacgmv2", "aacgmv2_mlt", or "geographic"
AACGM_HEIGHT_KM = 300  # same conversion altitude for data, radar markers and coastlines
AACGM_EXTENT = [-100, 50, 35, 85]  # magnetic longitude, magnetic latitude
MLT_EXTENT = [-180, 180, 40, 90]  # longitude = MLT hours * 15, wrapped to +/-180


In [ ]:
import bz2
import datetime as dt
from functools import lru_cache
from pathlib import Path
import os
import warnings

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "event_list.py").exists():
    raise RuntimeError("Start the notebook with the steve_stats project as the working directory.")
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".cache/matplotlib"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
import cartopy.crs as ccrs
from pyproj import Geod
import pydarn
import pydarnio
import aacgmv2
from sd_carto import SDCarto  # copied from meso_tracers/plotting_utils/sd_carto.py
from pydarn.utils.coordinates import gate2geographic_location
from IPython.display import display

start = pd.Timestamp(f"{EVENT_DATE} {START_TIME}")
end = pd.Timestamp(f"{EVENT_DATE} {END_TIME}")
if start == end:
    raise ValueError("Start and end must differ.")
if end < start:
    end += pd.Timedelta(days=1)
step = pd.Timedelta(TIME_DELTA)
if pd.isna(step) or step <= pd.Timedelta(0):
    raise ValueError("TIME_DELTA must be positive.")
if SCATTER not in {"both", "ground", "ionospheric"}:
    raise ValueError("SCATTER must be both, ground, or ionospheric.")
radars = list(dict.fromkeys(str(r).strip().lower() for r in RADARS))
if not radars or VELOCITY_LIMIT <= 0 or VECTOR_SECONDS <= 0:
    raise ValueError("Select radars and positive velocity/display scales.")
data_root = (PROJECT_ROOT / DATA_ROOT).resolve()
if COORDINATES not in {"geographic", "aacgmv2", "aacgmv2_mlt"}:
    raise ValueError("COORDINATES must be geographic, aacgmv2, or aacgmv2_mlt.")
output_dir = PROJECT_ROOT / OUTPUT_ROOT / start.strftime("%Y%m%d") / SCATTER / COORDINATES
windows = [(t, min(t + step, end)) for t in pd.date_range(start, end, freq=step, inclusive="left")]
print(f"{len(windows)} windows: {start} to {end} UT; scatter={SCATTER}")


In [ ]:
def record_time(record):
    return dt.datetime(*(int(record[f"time.{key}"]) for key in
                         ("yr", "mo", "dy", "hr", "mt", "sc")),
                       microsecond=int(record.get("time.us", 0)))


def read_records(path):
    with bz2.open(path, "rb") as stream:
        raw = stream.read()
    if hasattr(pydarn, "SuperDARNRead"):
        return pydarn.SuperDARNRead(raw, True).read_fitacf()
    records = pydarnio.read_fitacf(raw)
    return records[0] if isinstance(records, tuple) else records


@lru_cache(maxsize=None)
def gate_position(station, radar, day, beam, gate, frang, rsep):
    # pyDARN's geometry reads hardware from its registry. Restore after use.
    radar_info = pydarn.SuperDARNRadars.radars[pydarn.RadarID(station)]
    original = radar_info.hardware_info
    hardware = pydarn.read_hdw_file(radar, date=dt.datetime.fromisoformat(day))
    try:
        pydarn.SuperDARNRadars.radars[pydarn.RadarID(station)] = radar_info._replace(hardware_info=hardware)
        lat, lon = gate2geographic_location(
            stid=pydarn.RadarID(station), beam=beam, range_gate=gate,
            frang=frang, rsep=rsep, center=True, height=300)
    finally:
        pydarn.SuperDARNRadars.radars[pydarn.RadarID(station)] = radar_info
    return float(lat), float(lon), hardware.geographic.lat, hardware.geographic.lon


rows, inventory = [], []
for day in pd.date_range(start.normalize(), (end - pd.Timedelta(nanoseconds=1)).normalize(), freq="D"):
    for radar in radars:
        folder = data_root / day.strftime("%Y%m%d") / radar
        files = sorted(folder.glob("*.fitacf3.bz2")) or sorted(folder.glob("*.fitacf.bz2"))
        inventory.append({"date": str(day.date()), "radar": radar, "files": len(files)})
        if not files:
            warnings.warn(f"No files for {radar} on {day.date()} in {folder}")
        for path in files:
            # Read every daily file: avoids assuming archive file duration.
            for record in read_records(path):
                time = record_time(record)
                if not start <= time < end or "slist" not in record:
                    continue
                gates = record["slist"]
                velocities, flags = record.get("v", []), record.get("gflg", [])
                if not (len(gates) == len(velocities) == len(flags)):
                    warnings.warn(f"Skipping inconsistent arrays in {path.name} at {time}")
                    continue
                for gate, velocity, flag in zip(gates, velocities, flags):
                    if flag not in (0, 1) or not np.isfinite(velocity):
                        continue
                    lat, lon, radar_lat, radar_lon = gate_position(
                        int(record["stid"]), radar, time.date().isoformat(),
                        int(record["bmnum"]), int(gate), float(record["frang"]), float(record["rsep"]))
                    if np.isfinite(lat) and np.isfinite(lon):
                        rows.append((time, radar, lat, lon, radar_lat, radar_lon, float(velocity), int(flag)))
observations = pd.DataFrame(rows, columns=["time", "radar", "lat", "lon", "radar_lat", "radar_lon", "velocity", "gflg"])
display(pd.DataFrame(inventory))
if observations.empty:
    raise RuntimeError("No finite LOS observations in this interval. Check the inventory, date and times.")
if SCATTER != "both":
    observations = observations.loc[observations.gflg.eq(1 if SCATTER == "ground" else 0)].copy()
print(f"Selected {len(observations):,} observations.")


In [ ]:
geod = Geod(ellps="WGS84")
if COORDINATES == "geographic":
    projection = ccrs.LambertConformal(central_longitude=-95, central_latitude=55)
    extent = MAP_EXTENT
elif COORDINATES == "aacgmv2":
    projection = ccrs.NorthPolarStereo(central_longitude=-25)
    extent = AACGM_EXTENT
else:
    projection = ccrs.NorthPolarStereo(central_longitude=0)
    extent = MLT_EXTENT


def map_coordinates(lon, lat, timestamp):
    """Convert geographic points; the same timestamp is used throughout a panel."""
    lon, lat = np.atleast_1d(lon).astype(float), np.atleast_1d(lat).astype(float)
    if COORDINATES == "geographic":
        return lon, lat
    mlat, mlon, _ = aacgmv2.convert_latlon_arr(
        lat, lon, AACGM_HEIGHT_KM, timestamp.to_pydatetime(), method_code="G2A")
    if COORDINATES == "aacgmv2_mlt":
        mlon = np.asarray(aacgmv2.convert_mlt(mlon, timestamp.to_pydatetime())) * 15
    return (np.asarray(mlon) + 180) % 360 - 180, np.asarray(mlat)

geo = ccrs.PlateCarree()
norm = Normalize(-VELOCITY_LIMIT, VELOCITY_LIMIT)
cmap = plt.get_cmap("RdBu_r")
if SAVE_MAPS:
    output_dir.mkdir(parents=True, exist_ok=True)
summary = []
columns = min(GRID_COLUMNS, len(windows))
if columns < 1:
    raise ValueError("GRID_COLUMNS must be a positive integer.")
row_count = (len(windows) + columns - 1) // columns
fig = plt.figure(figsize=(6 * columns, 5 * row_count), layout="constrained")
axes = []
for index, (left, right) in enumerate(windows):
    panel_time = left + (right - left) / 2
    ax = fig.add_subplot(row_count, columns, index + 1, projection="sdcarto",
                         map_projection=projection, plot_date=panel_time.to_pydatetime(),
                         coords="geo" if COORDINATES == "geographic" else COORDINATES)
    ax.aacgm_height = AACGM_HEIGHT_KM
    axes.append(ax)
    frame = observations.loc[(observations.time >= left) & (observations.time < right)]
    ax.set_extent(extent, crs=geo)
    if COASTLINES:
        ax.coastlines(resolution="110m", linewidth=0.6)
    grid = ax.gridlines(draw_labels={"bottom": "x", "left": "y"}, linewidth=0.4, alpha=0.4)
    if COORDINATES == "aacgmv2_mlt":
        from matplotlib.ticker import FuncFormatter
        grid.xformatter = FuncFormatter(lambda value, pos: f"{(value / 15) % 24:g}h")
    for flag, marker, size in [(0, ".", 10), (1, "o", 12)]:
        subset = frame.loc[frame.gflg.eq(flag)]
        if subset.empty:
            continue
        lon, lat, velocity = (subset[c].to_numpy() for c in ("lon", "lat", "velocity"))
        toward, _, _ = geod.inv(lon, lat, subset.radar_lon.to_numpy(), subset.radar_lat.to_numpy())
        bearing = toward + np.where(velocity >= 0, 0, 180)
        tip_lon, tip_lat, _ = geod.fwd(lon, lat, bearing, np.abs(velocity) * VECTOR_SECONDS)
        lon, lat = map_coordinates(lon, lat, panel_time)
        tip_lon, tip_lat = map_coordinates(tip_lon, tip_lat, panel_time)
        base_xy = projection.transform_points(geo, lon, lat)
        tip_xy = projection.transform_points(geo, tip_lon, tip_lat)
        ax.quiver(base_xy[:, 0], base_xy[:, 1], tip_xy[:, 0] - base_xy[:, 0],
                  tip_xy[:, 1] - base_xy[:, 1], velocity, cmap=cmap, norm=norm,
                  angles="xy", scale_units="xy", scale=1, width=0.0018)
        ax.scatter(lon, lat, c=velocity, cmap=cmap, norm=norm, marker=marker,
                   s=size, transform=geo, linewidths=0.4, zorder=3)
    # Include available selected radar sites even in windows without returns.
    sites = pd.DataFrame(rows, columns=["time", "radar", "lat", "lon", "radar_lat", "radar_lon", "velocity", "gflg"])
    for radar, site in sites.groupby("radar", sort=False):
        row = site.iloc[0]
        site_lon, site_lat = map_coordinates(row.radar_lon, row.radar_lat, panel_time)
        ax.plot(site_lon, site_lat, "k^", ms=5, transform=geo)
        ax.text(site_lon[0] + 1, site_lat[0], radar.upper(), transform=geo, fontsize=9)
    if frame.empty:
        ax.text(0.5, 0.5, "No selected scatter in this window", transform=ax.transAxes, ha="center")
    handles = [Line2D([], [], color="k", marker=m, linestyle="None", label=label)
               for flag, m, label in [(0, ".", "Ionospheric"), (1, "o", "Ground")]
               if SCATTER == "both" or (flag == 1) == (SCATTER == "ground")]
    ax.legend(handles=handles, loc="lower left")
    ax.set_title(f"{left:%H:%M} – {right:%H:%M} UT | {len(frame):,} returns", fontsize=11)
    summary.append({"start_UT": left, "end_UT": right, "returns": len(frame),
                    "radars_with_returns": ", ".join(sorted(frame.radar.unique()))})
fig.suptitle(f"{start:%Y-%m-%d %H:%M} – {end:%Y-%m-%d %H:%M} UT | {SCATTER} scatter | {COORDINATES}\n"
             f"{', '.join(r.upper() for r in radars)} | 1000 m/s arrow = {VECTOR_SECONDS:g} km; no time averaging",
             fontsize=15)
fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=axes,
             shrink=0.7, label="LOS velocity (m/s); positive toward radar", extend="both")
if SAVE_MAPS:
    grid_path = output_dir / f"los_grid_{start:%Y%m%dT%H%M%S}_{end:%Y%m%dT%H%M%S}.png"
    fig.savefig(grid_path, dpi=150, bbox_inches="tight")
    print(f"Saved grid: {grid_path}")
display(fig)
plt.close(fig)
display(pd.DataFrame(summary))
